# MobileNetV3 MTL (표정·나이) — 학습 & 튜닝

- 로그: `experiments/mtl_runs.jsonl`
- 설정: `configs/mtl/example.yaml`
- CSV: `image_path,expression_id,age`
- 가중치: `weights/face_mtl_mobilenetv3.pt`

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import yaml
from IPython.display import display
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

ROOT = Path.cwd().resolve()
for _ in range(5):
    if (ROOT / "config.py").is_file():
        break
    if ROOT.parent == ROOT:
        break
    ROOT = ROOT.parent
else:
    raise FileNotFoundError("emotion-bot 루트를 찾지 못했습니다.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import os

os.chdir(ROOT)

import config
from models.mtl_model import FaceAttributeMTL
from models.preprocess import IMAGENET_MEAN, IMAGENET_STD

EXPERIMENTS_LOG = ROOT / "experiments" / "mtl_runs.jsonl"
EXPERIMENTS_LOG.parent.mkdir(parents=True, exist_ok=True)

def log_run(params, metrics, notes=""):
    record = {
        "run_id": datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
        "params": params,
        "metrics": metrics,
        "notes": notes,
    }
    with EXPERIMENTS_LOG.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return record

def load_log():
    if not EXPERIMENTS_LOG.is_file() or EXPERIMENTS_LOG.stat().st_size == 0:
        return pd.DataFrame()
    rows = [json.loads(ln) for ln in EXPERIMENTS_LOG.read_text(encoding="utf-8").splitlines() if ln.strip()]
    flat = []
    for r in rows:
        row = {"run_id": r["run_id"]}
        row.update({f"p_{k}": v for k, v in r.get("params", {}).items()})
        row.update({f"m_{k}": v for k, v in r.get("metrics", {}).items()})
        flat.append(row)
    return pd.DataFrame(flat)

print("ROOT:", ROOT)

ROOT: C:\Users\moonjintae\projects\emotion bot(cursor ver)


## 1. 파라미터 & 데이터 경로

In [ ]:
def _build_transforms(input_size: int, train: bool) -> transforms.Compose:
    steps = [transforms.Resize((input_size, input_size))]
    if train:
        steps += [
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.15, contrast=0.15),
        ]
    steps += [
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
    return transforms.Compose(steps)


class FaceAttributeDataset(Dataset):
    def __init__(self, csv_path, root, input_size=224, train: bool = True):
        self.df = pd.read_csv(csv_path)
        self.root = Path(root)
        self.transform = _build_transforms(input_size, train=train)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.root / str(row["image_path"])).convert("RGB")
        return self.transform(img), int(row["expression_id"]), float(row["age"])


YAML_CFG = ROOT / "configs" / "mtl" / "example.yaml"
cfg = yaml.safe_load(YAML_CFG.read_text(encoding="utf-8")) if YAML_CFG.is_file() else {}
TRAIN_PARAMS = {
    "run_name": cfg.get("run_name", "mtl_nb_001"),
    "epochs": cfg.get("epochs", 15),
    "batch_size": cfg.get("batch_size", 32),
    "lr": cfg.get("lr", 1e-3),
    "age_weight": cfg.get("age_weight", 0.01),
    "variant": cfg.get("variant", config.MOBILENET_VARIANT),
    "weight_decay": cfg.get("weight_decay", 1e-4),
    "dropout": cfg.get("dropout", 0.2),
    "patience": cfg.get("patience", 4),
}
def _resolve_data_path(p: str | Path) -> Path:
    path = Path(p)
    return path if path.is_absolute() else (ROOT / path).resolve()


TRAIN_CSV = _resolve_data_path(cfg.get("train_csv", config.MTL_TRAIN_CSV))
VAL_CSV = (
    _resolve_data_path(cfg["val_csv"])
    if cfg.get("val_csv")
    else _resolve_data_path(config.MTL_VAL_CSV)
)
DATA_ROOT = _resolve_data_path(cfg.get("data_root", config.MTL_DATA_DIR))
OUTPUT = config.MTL_WEIGHTS

print("cwd:", Path.cwd())
print("train:", TRAIN_CSV, TRAIN_CSV.is_file(), f"({len(pd.read_csv(TRAIN_CSV))} rows)" if TRAIN_CSV.is_file() else "")
print("val:", VAL_CSV, VAL_CSV.is_file() if VAL_CSV else None, f"({len(pd.read_csv(VAL_CSV))} rows)" if VAL_CSV and VAL_CSV.is_file() else "")
print("data_root:", DATA_ROOT, DATA_ROOT.is_dir())
if not TRAIN_CSV.is_file():
    print("→ python scripts/prepare_mtl_from_roboflow_emotion.py --age-backend opencv")
TRAIN_PARAMS

## 2. 학습

In [ ]:
RUN_TRAIN = True

if RUN_TRAIN and not TRAIN_CSV.is_file():
    raise FileNotFoundError(f"train CSV 없음: {TRAIN_CSV}")

if RUN_TRAIN:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)
    model = FaceAttributeMTL(
        num_expressions=config.NUM_EXPRESSIONS,
        variant=TRAIN_PARAMS["variant"],
        pretrained_backbone=True,
        dropout=TRAIN_PARAMS["dropout"],
    ).to(device)
    train_loader = DataLoader(
        FaceAttributeDataset(TRAIN_CSV, DATA_ROOT, config.INPUT_SIZE, train=True),
        batch_size=TRAIN_PARAMS["batch_size"],
        shuffle=True,
        num_workers=0,
    )
    val_loader = None
    if VAL_CSV and VAL_CSV.is_file():
        val_loader = DataLoader(
            FaceAttributeDataset(VAL_CSV, DATA_ROOT, config.INPUT_SIZE, train=False),
            batch_size=TRAIN_PARAMS["batch_size"],
            shuffle=False,
            num_workers=0,
        )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRAIN_PARAMS["lr"],
        weight_decay=TRAIN_PARAMS["weight_decay"],
    )
    ce, mse = nn.CrossEntropyLoss(), nn.MSELoss()
    best_val = float("inf")
    epochs_no_improve = 0
    patience = int(TRAIN_PARAMS["patience"])

    for epoch in range(1, TRAIN_PARAMS["epochs"] + 1):
        model.train()
        total = 0.0
        for images, expr_ids, ages in train_loader:
            images, expr_ids = images.to(device), expr_ids.to(device)
            ages = ages.to(device, dtype=torch.float32)
            optimizer.zero_grad(set_to_none=True)
            expr_logits, pred_ages = model(images)
            loss = ce(expr_logits, expr_ids) + TRAIN_PARAMS["age_weight"] * mse(
                pred_ages, ages
            )
            loss.backward()
            optimizer.step()
            total += float(loss.item()) * images.size(0)
        train_loss = total / len(train_loader.dataset)
        msg = f"epoch {epoch} train_loss={train_loss:.4f}"

        if val_loader:
            model.eval()
            vtotal = 0.0
            with torch.inference_mode():
                for images, expr_ids, ages in val_loader:
                    images, expr_ids = images.to(device), expr_ids.to(device)
                    ages = ages.to(device, dtype=torch.float32)
                    expr_logits, pred_ages = model(images)
                    vloss = ce(expr_logits, expr_ids) + TRAIN_PARAMS["age_weight"] * mse(
                        pred_ages, ages
                    )
                    vtotal += float(vloss.item()) * images.size(0)
            val_loss = vtotal / len(val_loader.dataset)
            msg += f" val_loss={val_loss:.4f}"
            if val_loss < best_val:
                best_val = val_loss
                epochs_no_improve = 0
                OUTPUT.parent.mkdir(parents=True, exist_ok=True)
                torch.save(model.state_dict(), OUTPUT)
                msg += " [saved]"
            else:
                epochs_no_improve += 1
                msg += f" (patience {epochs_no_improve}/{patience})"
        else:
            OUTPUT.parent.mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), OUTPUT)

        print(msg)
        if val_loader and epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch} (no val improvement for {patience} epochs)")
            break

    metrics = {
        "final_train_loss": train_loss,
        "best_val_loss": best_val if val_loader else None,
        "stopped_epoch": epoch,
    }
    record = log_run(TRAIN_PARAMS, metrics)
    print("weights:", OUTPUT)
    record

## 3. run 비교

In [ ]:
df = load_log()
if df.empty:
    print("experiments/mtl_runs.jsonl 비어 있음")
else:
    display(df.sort_values("run_id"))
    mcols = [c for c in df.columns if c.startswith("m_")]
    if mcols:
        df.plot(x="run_id", y=mcols, kind="bar", figsize=(8, 4), rot=45)
        plt.tight_layout()
        plt.show()